# ask_only

Ask a finished run's saved adapters new report questions. **No training.** A new question costs minutes
instead of a 3 to 4 hour run.

- The base model loads once. Each step applies that step's saved LoRA adapter (`checkpoints/<run_id>/decision/step-<N>`).
- Only `report_schema` (the question set and its settings) comes from `QUESTIONS_FROM`. The model, data folder,
  rule, persona, seeds and batch sizes are the run's own `results/<run_id>/settings.json`.
- The 1,000 decision trials are **not** asked again. The recovered rule each step is scored against is refitted
  from the run's saved `selections_decision_step-<N>.csv`.
- Writes only into `results/<run_id>/ask_only/<NAME>/`. Never appends to `results/results.csv` and never changes a
  file of the run.

Outputs per label folder:

| file | what |
|---|---|
| `reports_raw_decision_step-<N>.jsonl` | every reply, with run_id, step, question_name, trial_index, trial_kind, show_neither, reply_prefix, prompt and both options' values |
| `step_answers.md` / `.csv` | every question's answers per step |
| `faithfulness_by_trial_kind.csv` | the weights cosine per trial kind, over all five values and over the four unscreened weights (`cos_recovered_4`, the one to quote) |
| `cutoff_flags.csv` / `_summary.csv` | per cut reply: dimension right, distance from the cut, copies of shown values, range ends, by trial kind |
| `reply_forms.csv` | weight replies per trial kind: all five -100, all five 0, screened attribute at 100, bare letter or number, other |
| `timing.csv` | seconds per step |

Code: `src/ask_only.py`. Tests: `python -m tests.test_ask_only`.

## Settings

Edit this cell only, then run the cells below it.

In [ ]:
RUN = "20261110-112915"            # a run id, or its timestamp prefix
STEPS = [1000]                     # saved decision steps
QUESTIONS_FROM = "configs/minimum_threshold_one_persona/base/persona_b/one_persona_probes_b.json"   # take report_schema from this file; None = the run's own
ONLY = ["main", "main_trial_first", "screen_index", "screen_cut_units", "main_report_units",
        "main_threshold_met", "main_threshold_not_met", "screen_index_no_trial", "screen_cut_units_no_trial",
        "screen_cut_percent_no_trial", "main_no_trial", "cut_probe"]   # None = every question the file asks
SHOW_NEITHER = True                # True / False, or None to keep the file's setting
REPLY_PREFIX = "{"                 # a string ("" = none), or None to keep the file's setting
PER_KIND = 50                      # N trials of each kind (both pass / one fails / both fail); None = the original 50
NAME = "probes-c-prefix-perkind50" # output folder: results/<run_id>/ask_only/<NAME>/

## Set up

In [ ]:
import os, sys, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "ask_only.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("run this notebook from inside the project folder")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)

from src import ask_only as A
from src import config as C
from src import data as D
print("project:", ROOT)

## Which adapters are saved

In [ ]:
run_dir = A.find_run(RUN)
print(run_dir.name)
print("saved steps:", A.available_steps(run_dir.name))
missing = [s for s in STEPS if s not in A.available_steps(run_dir.name)]
if missing:
    raise FileNotFoundError(f"steps {missing} have no adapter locally; see the Hub repo adamchanna/bdp-plunkett-qwen3")

## Plan: replies and time

`SECONDS_PER_REPLY` is measured by the smoke cell below (or by any earlier pass: see `timing.csv`). If the plan is
over 4 hours, stop and decide before running.

In [ ]:
SECONDS_PER_REPLY = None    # set from a measured pass, e.g. seconds / replies of one step

cfg = A.run_settings(run_dir, QUESTIONS_FROM, ONLY, SHOW_NEITHER, REPLY_PREFIX)
data = D.load_data(cfg, C.resolve(cfg).rule)
per_step = A.reply_count(cfg, data, PER_KIND)
print(f"{per_step} replies per step x {len(STEPS)} steps = {per_step * len(STEPS)} replies")
if SECONDS_PER_REPLY:
    print(f"about {per_step * len(STEPS) * SECONDS_PER_REPLY / 3600:.2f} h")

## Ask

In [ ]:
LOADED = globals().get("LOADED")    # the base model, kept across runs of this cell
out_dir, timing, LOADED = A.ask(RUN, STEPS, QUESTIONS_FROM, ONLY, SHOW_NEITHER, REPLY_PREFIX, PER_KIND, NAME, loaded=LOADED)
print(out_dir)
pd.DataFrame(timing)

## Results

In [ ]:
A.report_table(out_dir)

In [ ]:
fk = pd.read_csv(out_dir / "faithfulness_by_trial_kind.csv")
fk[fk["question"].isin(["main", "main_trial_first"])][
    ["step", "question", "kind", "n", "cos_recovered_4", "cos_recovered", "share_all_-100", "share_screened_100", "mean_report", "recovered"]]

In [ ]:
cf = out_dir / "cutoff_flags_summary.csv"
pd.read_csv(cf) if cf.exists() else "no cut questions asked"

In [ ]:
print((out_dir / "step_answers.md").read_text(encoding="utf-8"))

## Check 1: reproduction

The run's own questions, its original 50 trials, `show_neither` off and no prefix. The replies must match the
run's saved `reports_raw_decision_step-<N>.jsonl`. If the match rate is not near 100%, stop and find out why.

In [ ]:
REPRO_RUN, REPRO_STEP = "20261110-112915", 1000
out, _, LOADED = A.ask(REPRO_RUN, [REPRO_STEP], None, None, False, "", None, f"reproduce-step-{REPRO_STEP}", loaded=globals().get("LOADED"))
rates, extra = A.reproduction(A.find_run(REPRO_RUN), out, REPRO_STEP)
print("asked but not in the saved file:", extra)
rates

## Passes

Each job is one call of `A.ask`. The base model stays loaded across jobs (all four runs are Qwen3-8B).

**Pass 1** (the baseline for the new runs): B-new, B-old, A-new at steps 1000, 1500, 3000; `show_neither` on, prefix `{`,
50 per kind; questions from `one_persona_probes_b.json` for the B runs and `one_persona_probes_a.json` for A-new.

**Pass 2** (does the C line or the `{` change the reply forms?): the same three runs, step 1000, `main` only, 50 per kind,
the three other combinations of `show_neither` and the prefix.

In [ ]:
RUNS = {"B-new": ("20261110-112915", "configs/minimum_threshold_one_persona/base/persona_b/one_persona_probes_b.json"),
        "B-old": ("20261109-214726", "configs/minimum_threshold_one_persona/base/persona_b/one_persona_probes_b.json"),
        "A-new": ("20261110-211307", "configs/minimum_threshold_one_persona/base/persona_a/one_persona_probes_a.json")}
PASS1_ONLY = ["main", "main_trial_first", "screen_index", "screen_cut_units", "main_report_units",
              "main_threshold_met", "main_threshold_not_met", "screen_index_no_trial", "screen_cut_units_no_trial",
              "screen_cut_percent_no_trial", "main_no_trial", "cut_probe"]
JOBS = []
for label, (run, qfile) in RUNS.items():
    JOBS.append(dict(run=run, steps=[1000, 1500, 3000], questions_from=qfile, only=PASS1_ONLY,
                     show_neither=True, reply_prefix="{", per_kind=50, name="pass1-c-on-prefix"))
for label, (run, qfile) in RUNS.items():
    for neither, prefix, name in ((False, "", "pass2-c-off-noprefix"), (True, "", "pass2-c-on-noprefix"),
                                  (False, "{", "pass2-c-off-prefix")):
        JOBS.append(dict(run=run, steps=[1000], questions_from=qfile, only=["main"],
                         show_neither=neither, reply_prefix=prefix, per_kind=50, name=name))

total = 0
for j in JOBS:
    cfg_j = A.run_settings(A.find_run(j["run"]), j["questions_from"], j["only"], j["show_neither"], j["reply_prefix"])
    n = A.reply_count(cfg_j, D.load_data(cfg_j, C.resolve(cfg_j).rule), j["per_kind"]) * len(j["steps"])
    total += n
    print(f"{j['name']:<24} {j['run']}  steps {j['steps']}: {n} replies")
print(f"total {total} replies" + (f", about {total * SECONDS_PER_REPLY / 3600:.2f} h" if SECONDS_PER_REPLY else ""))

In [ ]:
RUN_PASSES = False     # True once the plan above is agreed
if RUN_PASSES:
    for j in JOBS:
        t = time.time()
        out, timing, LOADED = A.ask(j["run"], j["steps"], j["questions_from"], j["only"], j["show_neither"],
                                    j["reply_prefix"], j["per_kind"], j["name"], loaded=globals().get("LOADED"))
        print(f"{j['name']} {j['run']}: {time.time() - t:.0f}s", timing)

In [ ]:
# one table per run: every step and settings combination
for label, (run, _) in RUNS.items():
    rd = A.find_run(run)
    tables = []
    for folder in sorted((rd / "ask_only").glob("pass*")):
        t = A.report_table(folder)
        t.insert(0, "settings", folder.name)
        tables.append(t)
    if tables:
        print(f"== {label} ({rd.name})")
        display(pd.concat(tables, ignore_index=True))